# 🏥 Day 1: Environment Setup & LLaVA-1.5-7B 4-bit NF4 Model Sanity Check
### Topic 4: Medical Diagnostic Support Visual Question Answering (VQA-Med-2019)
> **Sprint Milestones:** D1-T3 (Colab T4 Environment Setup) & D1-T4 (4-bit NF4 Forward Pass Verification)  
> **Target Hardware:** Google Colab Free Tier (1x NVIDIA T4 GPU, ~15 GB VRAM)  
> **Target Memory Footprint:** $\le 5.5\text{ GB}$ VRAM during base model loading

## 1. 🖥️ GPU & CUDA Environment Sanity Check

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Total GPU VRAM: {total_vram_gb:.2f} GB")
else:
    print("⚠️ WARNING: No CUDA GPU detected. Please go to Runtime -> Change runtime type -> T4 GPU.")

## 2. 📦 Install Required Dependencies
Install `transformers`, `peft`, `bitsandbytes`, `accelerate`, and visualization packages.

In [ ]:
!pip install -q --upgrade pip
!pip install -q transformers>=4.37.0 peft>=0.7.0 bitsandbytes>=0.41.0 accelerate>=0.26.0 torchvision pillow matplotlib

## 3. 📂 Mount Google Drive (Optional) & Setup Workspace

In [ ]:
import os
import sys

# Mount Google Drive if running in Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully.")
except ImportError:
    print("ℹ️ Running outside Google Colab. Skipping drive mount.")

## 4. ⚙️ Configure 4-bit NF4 Quantization (`BitsAndBytesConfig`)
We use **NormalFloat4 (NF4)** with **double quantization** and **FP16 compute dtype** to compress the 7B model from ~14GB down to **~4.5GB VRAM**.

In [ ]:
from transformers import BitsAndBytesConfig, AutoProcessor, LlavaForConditionalGeneration
import torch

MODEL_ID = "llava-hf/llava-1.5-7b-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print("BitsAndBytes 4-bit NF4 Configuration:")
print(bnb_config.to_dict())

## 5. 🚀 Load Base LLaVA-1.5-7B Model & Processor

In [ ]:
import time

print(f"Loading processor for {MODEL_ID}...")
processor = AutoProcessor.from_pretrained(MODEL_ID)

print(f"Loading 4-bit quantized model for {MODEL_ID}...")
start_time = time.time()
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
load_time = time.time() - start_time

print(f"✅ Model loaded in {load_time:.2f} seconds.")

## 6. 📊 VRAM Memory Footprint Audit
Verify that the loaded base model consumes $\le 5.5\text{ GB}$ of VRAM.

In [ ]:
if torch.cuda.is_available():
    vram_used_gb = torch.cuda.memory_allocated() / (1024**3)
    vram_max_gb = torch.cuda.max_memory_allocated() / (1024**3)
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    
    print(f"Current VRAM Allocated: {vram_used_gb:.2f} GB / {total_vram_gb:.2f} GB")
    print(f"Peak VRAM Allocated:    {vram_max_gb:.2f} GB / {total_vram_gb:.2f} GB")
    
    if vram_used_gb <= 5.5:
        print("\n🟢 PASSED: VRAM consumption is within the 5.5 GB target ceiling!")
    else:
        print(f"\n⚠️ CAUTION: VRAM consumption ({vram_used_gb:.2f} GB) exceeded 5.5 GB threshold.")

## 7. 🎯 Configure QLoRA Adapter (PEFT)
Configure LoRA with rank $r=16$, $\alpha=32$ on **all linear projection layers**.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=TARGET_MODULES,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# Attach PEFT adapter
peft_model = get_peft_model(model, lora_config)

# Compute and print trainable parameter ratio
trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in peft_model.parameters())
print(f"Trainable params: {trainable_params:,}")
print(f"All params:       {all_params:,}")
print(f"Trainable %:      {100.0 * trainable_params / all_params:.4f}%")
assert (trainable_params / all_params) < 0.02, "Trainable parameter ratio should be < 2% for efficient QLoRA!"

## 8. 🧪 Verify Forward Pass on Sample Medical Image

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

# Use a synthetic sample or an image from VQA-Med-2019 dataset
sample_image_path = "VQA-Med-2019/Val_images/synpic1001.jpg"

if os.path.exists(sample_image_path):
    image = Image.open(sample_image_path).convert("RGB")
else:
    print("Sample image path not found locally. Creating synthetic test radiology image...")
    # Synthetic medical image
    arr = np.zeros((336, 336, 3), dtype=np.uint8)
    arr[50:280, 50:280] = 120
    arr[100:230, 100:230] = 200
    image = Image.fromarray(arr)

# Display image
plt.figure(figsize=(4, 4))
plt.imshow(image)
plt.title("Input Query Image")
plt.axis("off")
plt.show()

# Formulate test question
question = "What imaging modality is shown in this image?"
prompt = f"USER: <image>\n{question}\nASSISTANT:"

inputs = processor(text=prompt, images=image, return_tensors="pt")
if torch.cuda.is_available():
    inputs = {k: v.to("cuda") for k, v in inputs.items()}

print("Running autoregressive forward pass...")
peft_model.eval()
with torch.no_grad():
    output_tokens = peft_model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=False,
        use_cache=True,
    )

input_len = inputs["input_ids"].shape[1]
gen_tokens = output_tokens[:, input_len:]
response = processor.batch_decode(gen_tokens, skip_special_tokens=True)[0].strip()

print("=" * 60)
print(f"Question: {question}")
print(f"Generated Response: {response}")
print("=" * 60)
print("🎉 All D1-T3 and D1-T4 validation criteria satisfied!")